# Simple Grovers algorithm - Simulator

In [1]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from qiskit.circuit.library import MCXGate


In [2]:
def oracle(circuit, qr, marked_state):
    n = len(marked_state)
    control_qubits = []
    for i, bit in enumerate(marked_state):
        if bit == '0':
            circuit.x(qr[i])  # Apply X gate to flip the qubit for '0'
        control_qubits.append(qr[i])
    
    # Apply multi-controlled Toffoli gate
    mcx_gate = MCXGate(len(control_qubits))
    circuit.append(mcx_gate, control_qubits + [qr[-1]])  # Target is the last qubit

    # Undo the X gates
    for i, bit in enumerate(marked_state):
        if bit == '0':
            circuit.x(qr[i])

In [3]:
def diffusion_operator(circuit, qr):
    n = len(qr)
    circuit.h(qr)  # Apply Hadamard gates
    circuit.x(qr)  # Apply X gates

    # Apply multi-controlled Toffoli gate
    mcx_gate = MCXGate(n - 1)
    circuit.append(mcx_gate, qr[:-1] + [qr[-1]])  # Target is the last qubit

    circuit.x(qr)  # Undo X gates
    circuit.h(qr)  # Undo Hadamard gates

In [7]:
# Define Grover's algorithm function
def grover(marked_state): 
    # Initialize a quantum register 
    # of n qubits 
    n = len(marked_state) 
    qr = QuantumRegister(n) 
    cr = ClassicalRegister(n) 
    circuit = QuantumCircuit(qr, cr) 
  
    # Apply the Hadamard gate 
    # to each qubit 
    circuit.h(qr) 
  
    # Repeat the following procedure 
    # O(sqrt(2 ^ n)) times 
    num_iterations = int(round((2 ** n) ** 0.5)) 
    for i in range(num_iterations): 
        # Apply the black box function f 
        # to the current state to mark 
        # the solution 
        oracle(circuit, qr, marked_state) 
        # Apply the Grover diffusion 
        # operator to amplify the amplitude 
        # of the marked solution 
        diffusion_operator(circuit, qr) 
  
    # Measure the state to obtain 
    # a solution x 
    circuit.measure(qr, cr) 
  
    # Run the circuit on a simulator 
    simulator = Aer.get_backend('qasm_simulator')
    compiled_circuit = transpile(circuit, simulator)
    sim_result = simulator.run(compiled_circuit).result()
    counts = sim_result.get_counts(circuit)
    # Get the most frequent state   
    x = max(counts, key=counts.get)
    # Convert the binary string to an integer
    x = int(x, 2)
    # Return the solution x
    return x 
  

In [8]:
# Apply Grover's algorithm to find the marked state
marked_state = '101'
result = grover(marked_state)
print(f"The marked state is {result}") 

CircuitError: 'duplicate qubit arguments'